In [2]:
from langchain_classic.agents.agent import AgentExecutor
from langchain_classic.agents.openai_tools.base import create_openai_tools_agent
from langchain_classic import hub

from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

from langchain_core.tools.retriever import create_retriever_tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

from langchain_openai import ChatOpenAI

import os

from dotenv import load_dotenv
load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [11]:
openai = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.1
)
 
# prompt = hub.pull(
#     "hwchase17/openai-functions-agent",
#     dangerously_pull_public_prompt=True
# )

api_wraper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)

wiki = WikipediaQueryRun(api_wrapper=api_wraper)

print(wiki.name)

wikipedia


In [13]:
loader = WebBaseLoader("https://news.naver.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200
).split_documents(docs)

vectordb = FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = vectordb.as_retriever()

print(retriever)

tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024223290980> search_kwargs={}


In [14]:
retriever_tool = create_retriever_tool(
    retriever, "naver_news_search",
    "네이버 뉴스 정보가 저장된 벡터 DB, 당일 기사에 대해 궁금하면 이 툴을 사용하세요!"
)

print(retriever_tool.name)

arxiv_wrapper = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=200,
    load_all_available_meta=False,
)

arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

print(arxiv.name)

naver_news_search
arxiv


In [16]:
from langsmith import Client

client = Client()
# pull_prompt 메서드를 통해 위험 인자를 직접 전달
prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent", 
    dangerously_pull_public_prompt=True
)

In [17]:
tools = [wiki, retriever_tool, arxiv]
agent = create_openai_tools_agent(
    llm=openai,
    tools=tools,
    prompt=prompt
)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_result = agent_executor.invoke({"input": "오늘 부동산 관련 주요 소식을 알려줘"})
print(agent_result)



> Entering new AgentExecutor chain...

Invoking: `naver_news_search` with `{'query': '부동산'}`


한동훈 32.7→38.2% 하정우 40.4→34.0% 엿새 만 반전? 박민식 20.9→23.3% [에이스리서치]


“반도체 폭락 사이클 잊지말라”…삼전·SK하닉 호황 속 美 전문가 경고











뉴시스
05월 26일 14:42

구독





[속보]합참 "北 미상 발사체는 근거리탄도미사일…서해상 수발 발사"



"두배로 베팅"…14만 개미, '삼전닉스 레버리지'에 줄섰다


삼성전자 잠정합의 투표 마감 D-1 참여율 90% 돌파…DX 노조는 '투표중지' 가처분


삼성전자, 내달 '제미나이·챗GPT·클로드' 외부 AI 도입 "업무생산성 향상"











월간산
05월 26일 08:53

구독





안데스 깊은 산속에도 자본주의는 칼처럼 작동했다 [산으로 간 남미]



생애 첫 등산화, 어떤 걸 살까? [등산왕]


퇴근 후 한 잔? 퇴근 후 한 산! [퇴근 산행 인왕산]


지리 '설악은 까칠해' vs 설악 '지리는 지루해' [산 대 산]











더팩트
05월 26일 14:20

구독





정용진, 스타벅스 '탱크데이' 리스크 관리 부실 인정 "누구도 문제 제기 없었다"



李 대통령 "안보는 경제강국 도약 핵심 토대"…핵잠·전작권 환수 속도 주문


민주 "평택을 단일화? 되겠나…조국과 골 깊어져"


성과급 치킨게임, 반도체 넘어 자동차·조선까지 번지나











아시아경제
05월 26일 14:35

구독





"오늘 올랐다고 웃을 때 아니다"…코스피 반등 뒤 도사린 폭락 경고



"엄마, 할머니집으로 '호캉스' 갈까?"…밥·청소 걱정 없고 3대가 편하게 모이는 조부모집[르포]


지능형 피싱 공격 증가…경찰, '해킹 전자우편' 대응 훈련


"한국 여자들처럼 살래요" 매출 272% 폭등… 맛집·학원 싹 쓸이하는 사람들 정체